# **Welcome to the notebook**

### Task 1 - Set up project environment

Installing the needed modules

In [4]:
!pip install openai==1.16.2 python-dotenv
!pip uninstall openai -y
!pip install --upgrade openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.1/267.1 kB 12.6 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.17.0
    Uninstalling openai-2.17.0:
      Successfully uninstalled openai-2.17.0
Found existing installation: openai 1.16.2
Uninstalling openai-1.16.2:
  Successfully uninstalled openai-1.16.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.2 MB/s eta 0:00:00


Importing the needed modules and setup the OpenAI API

In [5]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
from matplotlib import pyplot as plt
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer



Import our dataset

In [6]:
data = pd.read_csv('products_dataset.csv')
data

,product_id,title,description
0,P0,Men's 3X Large Carbon Heather Cotton/Polyester...,"This heavyweight, water-repellent hooded sweat..."
1,P1,Turmode 30 ft. RP TNC Female to RP TNC Male Ad...,If you need more length between your existing ...
2,P2,Large Tapestry Bolster Bed,Polyester cover resembling rich Italian tapest...
3,P3,16-Gauge-Sinks Vessel Sink in White with Faucet,It features a rectangle shape. This vessel set...
4,P4,Men's Crazy Horse 9'' Logger Boot - Steel Toe ...,This 9 in. black full grain leather logger boo...
...,...,...,...
1995,P1995,Dotty Black and White Black and White Wallpape...,"With a stylish monochrome look, this dotty wal..."
1996,P1996,Abrielle Brown/Light Gray 8 ft. x 10 ft. Orien...,The Abrielle collection features a stunning as...
1997,P1997,20 in. x 2-1/2 in. x 2-1/2 in. Polyurethane As...,"With Fypon balustrade systems, you can transfo..."
1998,P1998,1 gal. #P120-6 Diva Glam Flat Exterior Paint &...,BEHR PREMIUM PLUS Exterior Paint & Primer is a...


List of last 8 products recently viewed by the user.

In [7]:
searched_products_id = [
    'P1938',
    'P1970',
    'P1044',
    'P1838',
    'P1048',
    'P1017',
    'P1310',
    'P1444',
]

### Task 2 - Prepare the dataset

Let's label the data points that are recently veiwed.

In [8]:
data["product_status"]="notviewed"
data.loc[data.product_id.isin(searched_products_id), "product_status"] = "recentlyviewed"
data[data.product_status=="recentlyviewed"]

,product_id,title,description,product_status
1017,P1017,1 qt. #660D-7 Blackberry Farm Satin Enamel Int...,Love your space like never before with the hig...,recentlyviewed
1044,P1044,1 qt. #M360-4 Marjoram One-Coat Hide Eggshell ...,Introducing the best of BEHR Paint. Featuring ...,recentlyviewed
1048,P1048,5 gal. #640C-1 Hosta Flower Extra Durable Sati...,BEHR ULTRA SCUFF DEFENSE Stain-Blocking Paint ...,recentlyviewed
1310,P1310,5 gal. #180A-2 Romantic Morn Extra Durable Sem...,BEHR ULTRA SCUFF DEFENSE Stain-Blocking Paint ...,recentlyviewed
1444,P1444,5 gal. #PPU12-17 Cameroon Green Extra Durable ...,BEHR ULTRA SCUFF DEFENSE Stain-Blocking Paint ...,recentlyviewed
1838,P1838,5 gal. #N340-2 Dune Grass Extra Durable Satin ...,BEHR ULTRA SCUFF DEFENSE Stain-Blocking Paint ...,recentlyviewed
1938,P1938,1 gal. #HDC-SP16-10 Japanese Rose Garden Semi-...,Introducing the best of BEHR Paint. Featuring ...,recentlyviewed
1970,P1970,8 oz. #510C-3 Rivers Edge Semi-Gloss Enamel St...,Introducing the best of BEHR Paint. Featuring ...,recentlyviewed


Now let's combine the product `title` and `description` and store it into a column called `combined`.

In [9]:
data["combined"] = data["title"] + " " + data["description"]
data

,product_id,title,description,product_status,combined
0,P0,Men's 3X Large Carbon Heather Cotton/Polyester...,"This heavyweight, water-repellent hooded sweat...",notviewed,Men's 3X Large Carbon Heather Cotton/Polyester...
1,P1,Turmode 30 ft. RP TNC Female to RP TNC Male Ad...,If you need more length between your existing ...,notviewed,Turmode 30 ft. RP TNC Female to RP TNC Male Ad...
2,P2,Large Tapestry Bolster Bed,Polyester cover resembling rich Italian tapest...,notviewed,Large Tapestry Bolster Bed Polyester cover res...
3,P3,16-Gauge-Sinks Vessel Sink in White with Faucet,It features a rectangle shape. This vessel set...,notviewed,16-Gauge-Sinks Vessel Sink in White with Fauce...
4,P4,Men's Crazy Horse 9'' Logger Boot - Steel Toe ...,This 9 in. black full grain leather logger boo...,notviewed,Men's Crazy Horse 9'' Logger Boot - Steel Toe ...
...,...,...,...,...,...
1995,P1995,Dotty Black and White Black and White Wallpape...,"With a stylish monochrome look, this dotty wal...",notviewed,Dotty Black and White Black and White Wallpape...
1996,P1996,Abrielle Brown/Light Gray 8 ft. x 10 ft. Orien...,The Abrielle collection features a stunning as...,notviewed,Abrielle Brown/Light Gray 8 ft. x 10 ft. Orien...
1997,P1997,20 in. x 2-1/2 in. x 2-1/2 in. Polyurethane As...,"With Fypon balustrade systems, you can transfo...",notviewed,20 in. x 2-1/2 in. x 2-1/2 in. Polyurethane As...
1998,P1998,1 gal. #P120-6 Diva Glam Flat Exterior Paint &...,BEHR PREMIUM PLUS Exterior Paint & Primer is a...,notviewed,1 gal. #P120-6 Diva Glam Flat Exterior Paint &...


### Task 3 - Text embedding and visualization


Creating the text embedding vectors

In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")
data["text_embeddings"] = data["combined"].apply(
    lambda x: model.encode(x)
)
print(data["text_embeddings"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0       [-0.038377263, 0.081824824, -0.031021304, 0.02...
1       [-0.12242285, 0.011131762, -0.055760957, -0.02...
2       [-0.0017703436, 0.05645413, 0.0722064, 0.06515...
3       [-0.0025794487, 0.0761715, -0.0058366237, -0.0...
4       [-0.059780262, 0.029078474, 0.036028787, 0.033...
                              ...                        
1995    [-0.053201523, 0.066090755, 0.029283898, -0.01...
1996    [0.02767037, 0.017531527, 0.018372864, -0.0268...
1997    [0.040361486, 0.036026504, -0.040251706, 0.002...
1998    [-0.10739388, 0.044005197, 0.009033631, -0.023...
1999    [0.08616922, 0.107958116, 0.015794512, -0.0013...
Name: text_embeddings, Length: 2000, dtype: object


> We know that each vector has 384 dimensions. In order to be able to visualize the vectors in a scatter plot, we need to use Principal Component Analysis (PCA) to reduce the dimension from 384 to 2.

In [16]:
import numpy as np
embedding_matrix = np.vstack(data["text_embeddings"].values)

pca = PCA(n_components=2)
vector_2d = pca.fit_transform(embedding_matrix)
data['pca_one'] = vector_2d[:, 0]
data['pca_two'] = vector_2d[:, 1]


Now that we have the text embedding vectors in two dimensions, we can use them to create a 2D plot.

In [20]:
data
px.scatter(data,x='pca_one',y='pca_two',color="product_status")

### Task 4 - Find similar products

In [21]:
data.head()

,product_id,title,description,product_status,combined,text_embeddings,pca_one,pca_two
0,P0,Men's 3X Large Carbon Heather Cotton/Polyester...,"This heavyweight, water-repellent hooded sweat...",notviewed,Men's 3X Large Carbon Heather Cotton/Polyester...,"[-0.05433765494470784, 0.11154345219763988]",-0.054338,0.111543
1,P1,Turmode 30 ft. RP TNC Female to RP TNC Male Ad...,If you need more length between your existing ...,notviewed,Turmode 30 ft. RP TNC Female to RP TNC Male Ad...,"[-0.2010073979510767, -0.19740731411270077]",-0.201007,-0.197407
2,P2,Large Tapestry Bolster Bed,Polyester cover resembling rich Italian tapest...,notviewed,Large Tapestry Bolster Bed Polyester cover res...,"[-0.04263408108518246, 0.2777863347582759]",-0.042634,0.277786
3,P3,16-Gauge-Sinks Vessel Sink in White with Faucet,It features a rectangle shape. This vessel set...,notviewed,16-Gauge-Sinks Vessel Sink in White with Fauce...,"[-0.1612959227619355, -0.2413992078079432]",-0.161296,-0.241399
4,P4,Men's Crazy Horse 9'' Logger Boot - Steel Toe ...,This 9 in. black full grain leather logger boo...,notviewed,Men's Crazy Horse 9'' Logger Boot - Steel Toe ...,"[-0.3089773665566357, 0.02005610145293577]",-0.308977,0.020056


Get the data related to `recently_viewed` and `not_viewed` products

In [22]:
df_recentlyviewed = data[data.product_status=="recentlyviewed"]
df_notviewed = data[data.product_status=="notviewed"]
df_recentlyviewed

,product_id,title,description,product_status,combined,text_embeddings,pca_one,pca_two
1017,P1017,1 qt. #660D-7 Blackberry Farm Satin Enamel Int...,Love your space like never before with the hig...,recentlyviewed,1 qt. #660D-7 Blackberry Farm Satin Enamel Int...,"[0.5532878625591139, -0.0062202130431741765]",0.553288,-0.006220
1044,P1044,1 qt. #M360-4 Marjoram One-Coat Hide Eggshell ...,Introducing the best of BEHR Paint. Featuring ...,recentlyviewed,1 qt. #M360-4 Marjoram One-Coat Hide Eggshell ...,"[0.42705210937633675, 0.021405720771973533]",0.427052,0.021406
1048,P1048,5 gal. #640C-1 Hosta Flower Extra Durable Sati...,BEHR ULTRA SCUFF DEFENSE Stain-Blocking Paint ...,recentlyviewed,5 gal. #640C-1 Hosta Flower Extra Durable Sati...,"[0.5854968419206747, 0.04944287515930118]",0.585497,0.049443
1310,P1310,5 gal. #180A-2 Romantic Morn Extra Durable Sem...,BEHR ULTRA SCUFF DEFENSE Stain-Blocking Paint ...,recentlyviewed,5 gal. #180A-2 Romantic Morn Extra Durable Sem...,"[0.5780082930081762, 0.011772242366180248]",0.578008,0.011772
1444,P1444,5 gal. #PPU12-17 Cameroon Green Extra Durable ...,BEHR ULTRA SCUFF DEFENSE Stain-Blocking Paint ...,recentlyviewed,5 gal. #PPU12-17 Cameroon Green Extra Durable ...,"[0.5983576762433259, -0.016816430818425866]",0.598358,-0.016816
1838,P1838,5 gal. #N340-2 Dune Grass Extra Durable Satin ...,BEHR ULTRA SCUFF DEFENSE Stain-Blocking Paint ...,recentlyviewed,5 gal. #N340-2 Dune Grass Extra Durable Satin ...,"[0.5565011491139015, 0.03099255551303502]",0.556501,0.030993
1938,P1938,1 gal. #HDC-SP16-10 Japanese Rose Garden Semi-...,Introducing the best of BEHR Paint. Featuring ...,recentlyviewed,1 gal. #HDC-SP16-10 Japanese Rose Garden Semi-...,"[0.4710209060634957, 0.011106820282671648]",0.471021,0.011107
1970,P1970,8 oz. #510C-3 Rivers Edge Semi-Gloss Enamel St...,Introducing the best of BEHR Paint. Featuring ...,recentlyviewed,8 oz. #510C-3 Rivers Edge Semi-Gloss Enamel St...,"[0.47374606037188044, 0.04639468664021447]",0.473746,0.046395


Convert the embedding vectors to Numpy arrays

In [33]:
vectors_recently_viewed = [
    np.array(vector)
    for vector in df_recentlyviewed.text_embeddings
]
vectors_not_recently_viewed = [
    np.array(vector)
    for vector in df_notviewed.text_embeddings
]
vectors_recently_viewed


[array([ 0.55328786, -0.00622021]),
 array([0.42705211, 0.02140572]),
 array([0.58549684, 0.04944288]),
 array([0.57800829, 0.01177224]),
 array([ 0.59835768, -0.01681643]),
 array([0.55650115, 0.03099256]),
 array([0.47102091, 0.01110682]),
 array([0.47374606, 0.04639469])]

Find the similarity between each viewed product and all the unviewed products.

In [43]:
similarity_matrix = cosine_similarity(vectors_recently_viewed, vectors_not_recently_viewed)
top_ids = []
for row in similarity_matrix:
  top_id = np.argmax(row)
  top_ids.append(top_id)
most_similar_product_ids = list(df_notviewed.iloc[top_ids].product_id)
most_similar_product_ids

['P1327', 'P1556', 'P1266', 'P1648', 'P1536', 'P1274', 'P520', 'P378']

### Task 5 - Recommend products based on the searched products

Let's update the status of the top similar products to `recommended`.

In [45]:
data.loc[data.product_id.isin(most_similar_product_ids), "product_status"] = "recommended"
data[data.product_status=="recommended"]

,product_id,title,description,product_status,combined,text_embeddings,pca_one,pca_two
378,P378,1 qt. #N330-6 Lagoon Moss Extra Durable Semi-G...,BEHR ULTRA SCUFF DEFENSE Stain-Blocking Paint ...,recommended,1 qt. #N330-6 Lagoon Moss Extra Durable Semi-G...,"[0.5687356582108732, 0.05638532995459524]",0.568736,0.056385
520,P520,1 gal. #M410-5 Green Bank One-Coat Hide Matte ...,Introducing the best of BEHR Paint. Featuring ...,recommended,1 gal. #M410-5 Green Bank One-Coat Hide Matte ...,"[0.40942487009879547, 0.009338289261369159]",0.409425,0.009338
1266,P1266,1 gal. #260F-6 Smokey Topaz Semi-Gloss Enamel ...,Love your space like never before with the hig...,recommended,1 gal. #260F-6 Smokey Topaz Semi-Gloss Enamel ...,"[0.5680624833149791, 0.047306849654110694]",0.568062,0.047307
1274,P1274,1 gal. #N510-7 Blackout Gloss Enamel Interior/...,The BEHR Premium Porch and Patio Floor Paint E...,recommended,1 gal. #N510-7 Blackout Gloss Enamel Interior/...,"[0.3784630288053869, 0.021688746179423856]",0.378463,0.021689
1327,P1327,5 gal. #MQ4-44 Green Dynasty Extra Durable Egg...,BEHR ULTRA SCUFF DEFENSE Stain-Blocking Paint ...,recommended,5 gal. #MQ4-44 Green Dynasty Extra Durable Egg...,"[0.608454762716401, -0.006374248438783799]",0.608455,-0.006374
1536,P1536,1 qt. #130F-6 Brazil Nut Flat Exterior Paint &...,BEHR ULTRA Exterior Paint & Primer delivers ex...,recommended,1 qt. #130F-6 Brazil Nut Flat Exterior Paint &...,"[0.5663536178857516, -0.015980857384448263]",0.566354,-0.015981
1556,P1556,1 qt. #M480-1 Helium Extra Durable Satin Ename...,BEHR ULTRA SCUFF DEFENSE Stain-Blocking Paint ...,recommended,1 qt. #M480-1 Helium Extra Durable Satin Ename...,"[0.6132222403817458, 0.030367535383809822]",0.613222,0.030368
1648,P1648,1 Gal. Eggshell North Shore Interior Wall Pain...,Rust-Oleum Home Advanced Paint plus Primer Int...,recommended,1 Gal. Eggshell North Shore Interior Wall Pain...,"[0.4380139101373747, 0.008764131569047534]",0.438014,0.008764


Let's visualize the recommended products.

In [46]:
px.scatter(data,x='pca_one',y='pca_two',color="product_status",hover_data="title")